## 1. Installation Of Libraries

In [ ]:
!pip install numpy==1.23.5 Pillow==9.5.0 scikit-image==0.20.0 scipy==1.10.0

In [ ]:
import numpy
import skimage
import scipy
import PIL

print(f"NumPy: {numpy.__version__}")       
print(f"Scikit-Image: {skimage.__version__}") 
print(f"Pillow: {PIL.__version__}")        
print(f"SciPy: {scipy.__version__}")        

## 2. Setting up the Environment

In [ ]:
import os
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.colors as colors
import seaborn as sns
import numpy as np

from random import shuffle
from PIL import Image

from pycocotools.coco import COCO

## 3. Loading and Preparing the Dataset

In [ ]:

import numpy as np              
import os
from PIL import Image
from pycocotools.coco import COCO
from random import shuffle


ANNOTATION_FILE_TRAIN = '/kaggle/input/coco2014/captions/annotations/instances_train2014.json'
ANNOTATION_FILE_VAL = '/kaggle/input/coco2014/captions/annotations/instances_val2014.json'


classes = ['person']


coco_train = COCO(ANNOTATION_FILE_TRAIN)
catIds_train = coco_train.getCatIds(catNms=classes)
imgIds_train = coco_train.getImgIds(catIds=catIds_train)
imgDict_train = coco_train.loadImgs(imgIds_train)


coco_val = COCO(ANNOTATION_FILE_VAL)
catIds_val = coco_val.getCatIds(catNms=classes)
imgIds_val = coco_val.getImgIds(catIds=catIds_val)
imgDict_val = coco_val.loadImgs(imgIds_val)


print(len(imgIds_train), len(catIds_train))
print(len(imgIds_val), len(catIds_val))


shuffle(imgIds_train)
shuffle(imgIds_val)


imgIds_val = imgIds_val[0:2000]
imgIds_train = imgIds_train[0:60000]

train_images_person = ["COCO_train2014_{0:012d}.jpg".format(ids) for ids in imgIds_train]
val_images_person = ["COCO_val2014_{0:012d}.jpg".format(ids) for ids in imgIds_val]


print(len(train_images_person), len(val_images_person))


train_images_person = ["COCO_train2014_{0:012d}.jpg".format(ids) for ids in imgIds_train]


val_images_person = ["COCO_val2014_{0:012d}.jpg".format(ids) for ids in imgIds_val]


len(os.listdir("/kaggle/input/coco2014/val2014/val2014"))


!mkdir mask_train_2014


count = 0 


for ID in imgIds_train:

  
    file_path = "/kaggle/working/mask_train_2014/COCO_train2014_{0:012d}.jpg".format(ID)

  
    sampleImgIds = coco_train.getImgIds(imgIds=[ID])
    sampleImgDict = coco_train.loadImgs(sampleImgIds[np.random.randint(0, len(sampleImgIds))])[0]

    
    annIds = coco_train.getAnnIds(imgIds=sampleImgDict['id'], catIds=catIds_train, iscrowd=0)
    anns = coco_train.loadAnns(annIds)

   
    mask = coco_train.annToMask(anns[0])
    for i in range(len(anns)):
        mask = mask | coco_train.annToMask(anns[i])

    
    mask = Image.fromarray(mask * 255, mode="L")
    mask.save(file_path)
    count = count + 1


!mkdir mask_val_2014


count = 0


for ID in imgIds_val:

    
    file_path = "/kaggle/working/mask_val_2014/COCO_val2014_{0:012d}.jpg".format(ID)

    
    sampleImgIds = coco_val.getImgIds(imgIds=[ID])
    sampleImgDict = coco_val.loadImgs(sampleImgIds[np.random.randint(0, len(sampleImgIds))])[0]

    
    annIds = coco_val.getAnnIds(imgIds=sampleImgDict['id'], catIds=catIds_val, iscrowd=0)
    anns = coco_val.loadAnns(annIds)

    
    mask = coco_val.annToMask(anns[0])
    for i in range(len(anns)):
        mask = mask | coco_val.annToMask(anns[i])

  
    mask = Image.fromarray(mask * 255, mode="L")
    mask.save(file_path)

    count = count + 1

In [ ]:
!pip install numpy==1.26.4

In [ ]:
from tensorflow import keras
import numpy as np
from PIL import Image
import os

print(f"NumPy Version: {np.__version__}") 
print("Keras is ready!")

## 4. Custom Data Generator

In [ ]:
class CustomDataGenerator(keras.utils.Sequence):
    def __init__(self, images_path, masks_path, batch_size):
        """
        CustomDataGenerator class for generating batches of preprocessed images and masks.

        Args:
            images_path (str): Path to the directory containing the original images.
            masks_path (str): Path to the directory containing the corresponding masks.
            batch_size (int): Number of samples in each batch.

        Attributes:
            images_path (str): Path to the directory containing the original images.
            masks_path (str): Path to the directory containing the corresponding masks.
            batch_size (int): Number of samples in each batch.
            image_filenames (list): List of matching filenames between images and masks.
            mask_filenames (list): List of matching filenames between masks and images.
        """
        self.images_path = images_path
        self.masks_path = masks_path
        self.batch_size = batch_size
        self.image_filenames = self.get_matching_filenames()
        self.mask_filenames = self.get_matching_filenames()

    def get_matching_filenames(self):
        """
        Get the list of matching filenames between images and masks.

        Returns:
            list: List of matching filenames.
        """
        image_files = set([os.path.splitext(filename)[0] for filename in os.listdir(self.images_path)])
        mask_files = set([os.path.splitext(filename)[0] for filename in os.listdir(self.masks_path)])
        matching_files = list(image_files.intersection(mask_files))
        return matching_files

    def __len__(self):
        """
        Get the number of batches in the generator.

        Returns:
            int: Number of batches.
        """
        return int(np.ceil(len(self.image_filenames) / self.batch_size))

    def __getitem__(self, idx):
        """
        Get a batch of preprocessed images and masks.

        Args:
            idx (int): Batch index.

        Returns:
            tuple: Batch of preprocessed images and masks.
        """
        batch_filenames = self.image_filenames[idx * self.batch_size:(idx + 1) * self.batch_size]

        batch_images = []
        batch_masks = []

        for filename in batch_filenames:
            image_path = os.path.join(self.images_path, filename + '.jpg')
            mask_path = os.path.join(self.masks_path, filename + '.jpg')

            image = Image.open(image_path)
            mask = Image.open(mask_path)

            
            if image.size != mask.size:
                raise ValueError(f"Incompatible dimensions for image {image_path} and mask {mask_path}")

           
            image = image.resize((128, 128))
            mask = mask.resize((128, 128))

           
            preprocessed_image = np.array(image)
            preprocessed_mask = np.array(mask)

            
            if len(preprocessed_image.shape) == 3 and preprocessed_image.shape == (128, 128, 3):
            
                preprocessed_image = preprocessed_image / 255.0
                preprocessed_mask = preprocessed_mask / 255.0

                
                batch_images.append(preprocessed_image)
                batch_masks.append(preprocessed_mask)
        
        
        return np.array(batch_images), np.array(batch_masks)

In [ ]:

images_path = '/kaggle/input/coco2014/train2014/train2014'
masks_path = '/kaggle/working/mask_train_2014'
batch_size = 8


train_generator = CustomDataGenerator(images_path, masks_path, batch_size)

In [ ]:

print(len(os.listdir("/kaggle/input/coco2014/train2014/train2014")))


print(len(os.listdir("/kaggle/working/mask_train_2014")))


print(len(os.listdir("/kaggle/input/coco2014/val2014/val2014")))


print(len(os.listdir("/kaggle/working/mask_val_2014")))

In [ ]:
import matplotlib.pyplot as plt  
import numpy as np
import os
from PIL import Image

mask_filenames = [filename for filename in os.listdir(masks_path) if filename.endswith('.jpg')]


first_mask_filename = mask_filenames[0]
image_filename = os.path.splitext(first_mask_filename)[0] + '.jpg'


mask_image = Image.open(os.path.join(masks_path, first_mask_filename))
plt.subplot(1, 2, 1)
plt.imshow(mask_image)
plt.title('Mask Image')
plt.axis('off')


main_image = Image.open(os.path.join(images_path, image_filename))
plt.subplot(1, 2, 2)
plt.imshow(main_image)
plt.title('Main Image')
plt.axis('off')


print('Mask Image Shape:', np.array(mask_image).shape)
print('Main Image Shape:', np.array(main_image).shape)

plt.tight_layout()
plt.show()

## 5. Building the Model - UNet

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import *
import keras
from keras.layers import *
from keras.optimizers import *

def down_block(
    input_tensor,
    no_filters,
    kernel_size=(3, 3),
    strides=(1, 1),
    padding="same",
    kernel_initializer="he_normal",
    max_pool_window=(2, 2),
    max_pool_stride=(2, 2)
):
    conv = Conv2D(
        filters=no_filters,
        kernel_size=kernel_size,
        strides=strides,
        activation=None,
        padding=padding,
        kernel_initializer=kernel_initializer
    )(input_tensor)

    conv = BatchNormalization(scale=True)(conv)

    conv = Activation("relu")(conv)

    conv = Conv2D(
        filters=no_filters,
        kernel_size=kernel_size,
        strides=strides,
        activation=None,
        padding=padding,
        kernel_initializer=kernel_initializer
    )(conv)

    conv = BatchNormalization(scale=True)(conv)

   
    conv = Activation("relu")(conv)

    pool = MaxPooling2D(pool_size=max_pool_window, strides=max_pool_stride)(conv)

    return conv, pool

def bottle_neck(
    input_tensor,
    no_filters,
    kernel_size=(3, 3),
    strides=(1, 1),
    padding="same",
    kernel_initializer="he_normal"
):
    conv = Conv2D(
        filters=no_filters,
        kernel_size=kernel_size,
        strides=strides,
        activation=None,
        padding=padding,
        kernel_initializer=kernel_initializer
    )(input_tensor)

    conv = BatchNormalization(scale=True)(conv)

    conv = Activation("relu")(conv)

    conv = Conv2D(
        filters=no_filters,
        kernel_size=kernel_size,
        strides=strides,
        activation=None,
        padding=padding,
        kernel_initializer=kernel_initializer
    )(conv)

    conv = BatchNormalization(scale=True)(conv)

    conv = Activation("relu")(conv)

    return conv

def up_block(    
    input_tensor,
    no_filters,
    skip_connection, 
    kernel_size=(3, 3),
    strides=(1, 1),
    upsampling_factor = (2,2),
    max_pool_window = (2,2),
    padding="same",
    kernel_initializer="he_normal"):
    
    
    conv = Conv2D(
        filters = no_filters,
        kernel_size= max_pool_window,
        strides = strides,
        activation = None,
        padding = padding,
        kernel_initializer=kernel_initializer
    )(UpSampling2D(size = upsampling_factor)(input_tensor))
    
    conv = BatchNormalization(scale=True)(conv)

    conv = Activation("relu")(conv) 
    
    
    conv = concatenate( [skip_connection , conv]  , axis = -1)
    
    
    conv = Conv2D(
        filters=no_filters,
        kernel_size=kernel_size,
        strides=strides,
        activation=None,
        padding=padding,
        kernel_initializer=kernel_initializer
    )(conv)

    conv = BatchNormalization(scale=True)(conv)

    conv = Activation("relu")(conv)

    conv = Conv2D(
        filters=no_filters,
        kernel_size=kernel_size,
        strides=strides,
        activation=None,
        padding=padding,
        kernel_initializer=kernel_initializer
    )(conv)

    conv = BatchNormalization(scale=True)(conv)

    conv = Activation("relu")(conv)
    
    return conv


def output_block(input_tensor,
    padding="same",
    kernel_initializer="he_normal"
):
    
    conv = Conv2D(
        filters=2,
        kernel_size=(3,3),
        strides=(1,1),
        activation="relu",
        padding=padding,
        kernel_initializer=kernel_initializer
    )(input_tensor)
    
    
    conv = Conv2D(
        filters=1,
        kernel_size=(1,1),
        strides=(1,1),
        activation="sigmoid",
        padding=padding,
        kernel_initializer=kernel_initializer
    )(conv)
    
    
    return conv
    

def UNet(input_shape = (128,128,3)):
    
    filter_size = [64,128,256,512,1024]
    
    inputs = Input(shape = input_shape)
    
    d1 , p1 = down_block(input_tensor= inputs,
                         no_filters=filter_size[0],
                         kernel_size = (3,3),
                         strides=(1,1),
                         padding="same",
                         kernel_initializer="he_normal",
                         max_pool_window=(2,2),
                         max_pool_stride=(2,2))
    
    
    d2 , p2 = down_block(input_tensor= p1,
                         no_filters=filter_size[1],
                         kernel_size = (3,3),
                         strides=(1,1),
                         padding="same",
                         kernel_initializer="he_normal",
                         max_pool_window=(2,2),
                         max_pool_stride=(2,2))
    
    
    
    d3 , p3 = down_block(input_tensor= p2,
                         no_filters=filter_size[2],
                         kernel_size = (3,3),
                         strides=(1,1),
                         padding="same",
                         kernel_initializer="he_normal",
                         max_pool_window=(2,2),
                         max_pool_stride=(2,2))
    
    
    
    d4 , p4 = down_block(input_tensor= p3,
                         no_filters=filter_size[3],
                         kernel_size = (3,3),
                         strides=(1,1),
                         padding="same",
                         kernel_initializer="he_normal",
                         max_pool_window=(2,2),
                         max_pool_stride=(2,2))
    
    
    b = bottle_neck(input_tensor= p4,
                         no_filters=filter_size[4],
                         kernel_size = (3,3),
                         strides=(1,1),
                         padding="same",
                         kernel_initializer="he_normal")
    
    
    
    u4 = up_block(input_tensor = b,
                  no_filters = filter_size[3],
                  skip_connection = d4,
                  kernel_size=(3, 3),
                  strides=(1, 1),
                  upsampling_factor = (2,2),
                  max_pool_window = (2,2),
                  padding="same",
                  kernel_initializer="he_normal")
    
    u3 = up_block(input_tensor = u4,
                  no_filters = filter_size[2],
                  skip_connection = d3,
                  kernel_size=(3, 3),
                  strides=(1, 1),
                  upsampling_factor = (2,2),
                  max_pool_window = (2,2),
                  padding="same",
                  kernel_initializer="he_normal")
    
    
    u2 = up_block(input_tensor = u3,
                  no_filters = filter_size[1],
                  skip_connection = d2,
                  kernel_size=(3, 3),
                  strides=(1, 1),
                  upsampling_factor = (2,2),
                  max_pool_window = (2,2),
                  padding="same",
                  kernel_initializer="he_normal")
    
    
    u1 = up_block(input_tensor = u2,
                  no_filters = filter_size[0],
                  skip_connection = d1,
                  kernel_size=(3, 3),
                  strides=(1, 1),
                  upsampling_factor = (2,2),
                  max_pool_window = (2,2),
                  padding="same",
                  kernel_initializer="he_normal")
    
    
    
    output = output_block(input_tensor=u1 , 
                         padding = "same",
                         kernel_initializer= "he_normal")
    
    model = keras.models.Model(inputs = inputs , outputs = output)
    
    
    return model

## 6. Compiling and Training the Model

In [ ]:
import math 
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.metrics import MeanIoU, Precision, Recall, AUC, Accuracy
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import datetime
import os
import keras.backend as K
from tensorflow.keras.losses import Loss

images_path = '/kaggle/input/coco2014/val2014/val2014'
masks_path = '/kaggle/working/mask_val_2014'
batch_size = 32


val_generator = CustomDataGenerator(images_path, masks_path, batch_size)


train_images_path = '/kaggle/input/coco2014/train2014/train2014'
train_masks_path = '/kaggle/working/mask_train_2014'
train_generator = CustomDataGenerator(train_images_path, train_masks_path, batch_size)


train_steps = int(len(os.listdir(train_masks_path)) / batch_size)


checkpoint_path = '/kaggle/working/checkpoints/model_checkpoint.keras'
log_dir = '/kaggle/working/logs/'


epochs = 50


callbacks = [
    EarlyStopping(patience=10, restore_best_weights=True),
    ModelCheckpoint(checkpoint_path, save_best_only=True),
    ReduceLROnPlateau(factor=0.7, patience=5)
]


current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
log_dir = log_dir + current_time


metrics = [
    MeanIoU(num_classes=2, name='mean_io_u'),
    Precision(),
    Recall(),
    AUC(),
    'accuracy'
]

optimizer = Adam(learning_rate=1e-3)


model = UNet(input_shape=(128, 128, 3))
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=metrics)


history = model.fit(
    train_generator,
    validation_data=val_generator,
    steps_per_epoch=train_steps,
    epochs=epochs,
    batch_size=batch_size,
    callbacks=callbacks,
    verbose=1
)


model.save('/kaggle/working/final_model.keras')

## 7. Making Predictions and Visualizing Results

In [ ]:

sample_images, sample_masks = val_generator[4]


predictions = model.predict(sample_images)


threshold = 0.5  
thresholded_predictions = (predictions > threshold).astype(np.uint8)


idx = np.random.randint(0, sample_images.shape[0])


fig, axes = plt.subplots(1, 3, figsize=(12, 4))


axes[0].imshow(sample_images[idx])
axes[0].set_title('Sample Image')


axes[1].imshow(sample_masks[idx])
axes[1].set_title('Ground Truth Mask')


axes[2].imshow(thresholded_predictions[idx])
axes[2].set_title('Predicted Mask')


fig.suptitle('Sample Image, Ground Truth Mask, and Predicted Mask')


plt.tight_layout()


plt.show()

## 8. Visualizing Segmentation Results with Overlaid Masks

In [ ]:

for i in range(10):
    sample_images, sample_masks = val_generator[i]

    
    predictions = model.predict(sample_images)

    
    threshold = 0.5  
    thresholded_predictions = (predictions > threshold).astype(np.uint8)

   
    idx = np.random.randint(0, sample_images.shape[0])

    
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

   
    axes[0].imshow(sample_images[idx])
    axes[0].set_title('Sample Image')

    
    axes[1].imshow(sample_images[idx])
    axes[1].imshow(sample_masks[idx], cmap='jet', alpha=0.7)
    axes[1].set_title('Ground Truth Mask')

   
    axes[2].imshow(sample_images[idx])
    axes[2].imshow(thresholded_predictions[idx], cmap='jet', alpha=0.7)
    axes[2].set_title('Predicted Mask')

    
    fig.suptitle('Sample Image, Ground Truth Mask, and Predicted Mask')

    
    plt.tight_layout()

    
    plt.show()

## 9. Visualizing Training and Validation Performance Metrics (Loss, Accuracy, MloU)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np


sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
plt.rcParams['figure.dpi'] = 150 



if 'history' in globals():
    history_df = pd.DataFrame(history.history)
    best_epoch = history_df['val_loss'].idxmin() 

    
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))

   
    sns.lineplot(data=history_df[['loss', 'val_loss']], ax=axes[0], linewidth=2.5)
    axes[0].set_title('Model Convergence (Loss)', fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].axvline(best_epoch, color='green', linestyle='--', label=f'Best Epoch {best_epoch+1}')
    axes[0].legend()

    
    sns.lineplot(data=history_df[['accuracy', 'val_accuracy']], ax=axes[1], linewidth=2.5)
    axes[1].set_title('Classification Accuracy', fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].axvline(best_epoch, color='green', linestyle='--')

    
   
    miou_key = [k for k in history_df.columns if 'mean_io_u' in k and 'val' not in k]
    val_miou_key = [k for k in history_df.columns if 'mean_io_u' in k and 'val' in k]
    
    if miou_key and val_miou_key:
        sns.lineplot(data=history_df[[miou_key[0], val_miou_key[0]]], ax=axes[2], linewidth=2.5)
        axes[2].set_title('Intersection over Union (mIoU)', fontweight='bold')
        axes[2].set_xlabel('Epoch')
        axes[2].axvline(best_epoch, color='green', linestyle='--')

    plt.tight_layout()
    plt.show()
else:
    print("Could not find 'history' variable. Did you restart the kernel?")

## 10. Visualizing Training Dynamics and Quantitative Performance Metrics

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow as tf


sns.set_theme(style="ticks", context="paper", font_scale=1.4)
plt.rcParams['figure.dpi'] = 150
colors = ["#3498db", "#e74c3c", "#2ecc71"] # Blue, Red, Green for professional look


def plot_training_history(history):
    history_df = pd.DataFrame(history.history)
    best_epoch = history_df['val_loss'].idxmin()
    
    fig, axes = plt.subplots(1, 3, figsize=(22, 6))
    plt.suptitle("Model Training Performance", fontsize=20, fontweight='bold', y=1.05)

    
    sns.lineplot(data=history_df[['loss', 'val_loss']], ax=axes[0], palette=[colors[0], colors[1]], linewidth=3)
    axes[0].set_title('Loss Convergence (Binary Crossentropy)', fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].axvline(best_epoch, color=colors[2], linestyle='--', label=f'Best Epoch: {best_epoch+1}')
    axes[0].legend()
    axes[0].grid(True, linestyle='--', alpha=0.6)

    
    sns.lineplot(data=history_df[['accuracy', 'val_accuracy']], ax=axes[1], palette=[colors[0], colors[1]], linewidth=3)
    axes[1].set_title('Pixel Accuracy', fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].axvline(best_epoch, color=colors[2], linestyle='--')
    axes[1].grid(True, linestyle='--', alpha=0.6)

    
    miou_key = [k for k in history_df.columns if 'mean_io_u' in k and 'val' not in k]
    val_miou_key = [k for k in history_df.columns if 'mean_io_u' in k and 'val' in k]
    
    if miou_key:
        sns.lineplot(data=history_df[[miou_key[0], val_miou_key[0]]], ax=axes[2], palette=[colors[0], colors[1]], linewidth=3)
        axes[2].set_title('Intersection over Union (mIoU)', fontweight='bold')
        axes[2].set_xlabel('Epoch')
        axes[2].set_ylabel('mIoU Score')
        axes[2].axvline(best_epoch, color=colors[2], linestyle='--')
        axes[2].grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()


def evaluate_model_metrics(model, generator, steps=None):
    print("🚀 Calculating Pixel-wise Confusion Matrix... (This may take a moment)")
    
    
    total_cm = np.zeros((2, 2))
    
    if steps is None:
        steps = len(generator)
        
    for i in range(steps):
        
        imgs, true_masks = generator[i]
        preds = model.predict(imgs, verbose=0)
        
        
        pred_binary = (preds > 0.5).astype(int).flatten()
        true_binary = (true_masks > 0.5).astype(int).flatten()
        
        
        batch_cm = confusion_matrix(true_binary, pred_binary, labels=[0, 1])
        total_cm += batch_cm

   
    plt.figure(figsize=(8, 6))
    
    
    cm_normalized = total_cm.astype('float') / total_cm.sum(axis=1)[:, np.newaxis]
    
   
    group_counts = ["{0:0.0f}".format(value) for value in total_cm.flatten()]
    group_percentages = ["{0:.2%}".format(value) for value in cm_normalized.flatten()]
    labels = [f"{v1}\n{v2}" for v1, v2 in zip(group_counts, group_percentages)]
    labels = np.asarray(labels).reshape(2, 2)
    
    sns.heatmap(total_cm, annot=labels, fmt='', cmap='Blues', cbar=False,
                xticklabels=['Background (Pred)', 'Object (Pred)'],
                yticklabels=['Background (True)', 'Object (True)'])
    
    plt.title('Pixel-wise Confusion Matrix', fontweight='bold', fontsize=16)
    plt.ylabel('Actual Label')
    plt.xlabel('Predicted Label')
    plt.show()
    
    
    tn, fp, fn, tp = total_cm.ravel()
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    iou = tp / (tp + fp + fn) 

    print("\n" + "="*40)
    print("📊 FINAL MODEL TEST METRICS")
    print("="*40)
    print(f"{'Metric':<20} | {'Score':<10}")
    print("-" * 35)
    print(f"{'Pixel Accuracy':<20} | {accuracy:.4f}")
    print(f"{'Precision':<20} | {precision:.4f}")
    print(f"{'Recall (Sensitivity)':<20} | {recall:.4f}")
    print(f"{'F1-Score':<20} | {f1:.4f}")
    print(f"{'IoU (Jaccard)':<20} | {iou:.4f}")
    print("="*40)




if 'history' in globals():
    plot_training_history(history)


evaluate_model_metrics(model, val_generator, steps=10) 

## 11. Visual Comparison of Segmentation Results and Error Analysis

In [ ]:
def visualize_predictions_generator(generator, model, num_samples=3):
    
    sample_images, sample_masks = generator[0]
    
    
    preds = model.predict(sample_images)
    
    
    preds_binary = (preds > 0.5).astype(np.float32)
    
   
    num_samples = min(num_samples, len(sample_images))

   
    plt.figure(figsize=(20, 5 * num_samples))
    
    for i in range(num_samples):
       
        plt.subplot(num_samples, 4, i * 4 + 1)
        
        img_show = sample_images[i]
        plt.imshow(img_show)
        plt.title(f"Input Image #{i+1}", fontweight='bold')
        plt.axis('off')

        
        plt.subplot(num_samples, 4, i * 4 + 2)
       
        plt.imshow(np.squeeze(sample_masks[i]), cmap='gray')
        plt.title(f"Ground Truth #{i+1}", fontweight='bold')
        plt.axis('off')

        
        plt.subplot(num_samples, 4, i * 4 + 3)
        plt.imshow(np.squeeze(preds_binary[i]), cmap='magma') 
        plt.title(f"Prediction #{i+1}", fontweight='bold')
        plt.axis('off')

        
        
        diff = np.abs(np.squeeze(sample_masks[i]) - np.squeeze(preds_binary[i]))
        plt.subplot(num_samples, 4, i * 4 + 4)
        plt.imshow(diff, cmap='Reds', vmin=0, vmax=1)
        plt.title(f"Error Map (Red=Mistake)", fontweight='bold', color='darkred')
        plt.axis('off')

    plt.tight_layout()
    plt.show()


print("Generating high-quality visualizations...")
visualize_predictions_generator(val_generator, model, num_samples=5)

## 12. Applying the Model to an External Test Dataset

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm 


TEST_PATH = '/kaggle/input/test-seg'
OUTPUT_DIR = '/kaggle/working/inference_results'
IMG_SIZE = (128, 128) 


if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

def process_and_predict(model, source_dir, output_dir):
    print(f"📂 Scanning images in: {source_dir}")
    image_files = [f for f in os.listdir(source_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
    print(f"🚀 Found {len(image_files)} images. Starting inference...")

    
    sample_inputs = []
    sample_overlays = []

    for img_name in tqdm(image_files):
        img_path = os.path.join(source_dir, img_name)
        
       
        original_img = cv2.imread(img_path)
        original_img = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB) 
        
       
        input_img = cv2.resize(original_img, IMG_SIZE)
        
        
        input_tensor = input_img.astype(np.float32) / 255.0
        input_tensor = np.expand_dims(input_tensor, axis=0) 

        
        pred_mask = model.predict(input_tensor, verbose=0)
        
        
        pred_mask = (pred_mask > 0.5).astype(np.uint8) 
        pred_mask = np.squeeze(pred_mask) 

        
        colored_mask = np.zeros_like(input_img)
        colored_mask[:, :, 1] = 255 
        
       
        masked_region = cv2.bitwise_and(colored_mask, colored_mask, mask=pred_mask)
        
        
        overlay = cv2.addWeighted(input_img, 0.7, masked_region, 0.3, 0)

        
        save_path = os.path.join(output_dir, f"pred_{img_name}")
        
        cv2.imwrite(save_path, cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))
        
        
        if len(sample_inputs) < 5:
            sample_inputs.append(input_img)
            sample_overlays.append(overlay)

    print(f"\n✅ All predictions saved to: {output_dir}")
    return sample_inputs, sample_overlays


sample_inputs, sample_overlays = process_and_predict(model, TEST_PATH, OUTPUT_DIR)

## 13. Visualizing Inference: Overlaying Predictions on Test Images

In [ ]:
def visualize_test_results(inputs, overlays):
    plt.figure(figsize=(15, 3 * len(inputs)))
    
    for i in range(len(inputs)):
        
        plt.subplot(len(inputs), 2, i * 2 + 1)
        plt.imshow(inputs[i])
        plt.title(f"Test Image #{i+1}", fontweight='bold')
        plt.axis('off')
        
        
        plt.subplot(len(inputs), 2, i * 2 + 2)
        plt.imshow(overlays[i])
        plt.title(f"Model Prediction (Green Overlay)", fontweight='bold')
        plt.axis('off')
        
    plt.tight_layout()
    plt.show()


visualize_test_results(sample_inputs, sample_overlays)

## 14. Archiving and Downloading Inference Results of Test Data

In [ ]:

!zip -r inference_results.zip /kaggle/working/inference_results


print("\n================================================")
print("🎉 SUCCESS! Your results are ready.")
print("Look at the 'Output' sidebar on the right ->")
print("You should see 'inference_results.zip'.")
print("Click the dots (...) next to it and select 'Download'.")
print("================================================")

## 15. End to End Test Data Processing: Confidence Heatmaps and Multi - Channel Visualization

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm


TEST_PATH = '/kaggle/input/test-seg'
OUTPUT_DIR = '/kaggle/working/inference_results'
IMG_SIZE = (128, 128)  

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

def visualize_test_advanced(inputs, raw_preds, binary_preds, overlays):
    """
    Custom 4-Column Visualization for Test Data (No Ground Truth)
    1. Input | 2. Binary Mask | 3. Confidence Heatmap | 4. Overlay
    """
    num_samples = len(inputs)
    plt.figure(figsize=(20, 5 * num_samples))
    
    for i in range(num_samples):
        
        plt.subplot(num_samples, 4, i * 4 + 1)
        plt.imshow(inputs[i])
        plt.title(f"Test Image #{i+1}", fontweight='bold')
        plt.axis('off')

        
        plt.subplot(num_samples, 4, i * 4 + 2)
        plt.imshow(np.squeeze(binary_preds[i]), cmap='gray')
        plt.title(f"Predicted Mask (Binary)", fontweight='bold')
        plt.axis('off')

        
        
        plt.subplot(num_samples, 4, i * 4 + 3)
        plt.imshow(np.squeeze(raw_preds[i]), cmap='magma', vmin=0, vmax=1)
        plt.title(f"Confidence Heatmap", fontweight='bold')
        plt.axis('off')

       
        plt.subplot(num_samples, 4, i * 4 + 4)
        plt.imshow(overlays[i])
        plt.title(f"Final Overlay", fontweight='bold', color='green')
        plt.axis('off')

    plt.tight_layout()
    plt.show()

def process_test_data(model, source_dir, output_dir, max_display=5):
    print(f"📂 Scanning images in: {source_dir}")
    image_files = [f for f in os.listdir(source_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
    
    
    disp_inputs, disp_raw, disp_binary, disp_overlays = [], [], [], []
    
    print("🚀 Running Inference...")
    for img_name in tqdm(image_files):
        img_path = os.path.join(source_dir, img_name)
        
        
        original_img = cv2.imread(img_path)
        original_img = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)
        input_img = cv2.resize(original_img, IMG_SIZE)
        input_tensor = input_img.astype(np.float32) / 255.0
        input_tensor = np.expand_dims(input_tensor, axis=0)

        
        raw_pred = model.predict(input_tensor, verbose=0) 
        binary_pred = (raw_pred > 0.5).astype(np.uint8)   
        
        
        raw_pred = np.squeeze(raw_pred)
        binary_pred = np.squeeze(binary_pred)

        
        colored_mask = np.zeros_like(input_img)
        colored_mask[:, :, 1] = 255 # Green channel
        masked_region = cv2.bitwise_and(colored_mask, colored_mask, mask=binary_pred)
        overlay = cv2.addWeighted(input_img, 0.7, masked_region, 0.3, 0)

        
        save_path = os.path.join(output_dir, f"pred_{img_name}")
        cv2.imwrite(save_path, cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

        
        
        if len(disp_inputs) < max_display and np.sum(binary_pred) > 0:
            disp_inputs.append(input_img)
            disp_raw.append(raw_pred)
            disp_binary.append(binary_pred)
            disp_overlays.append(overlay)

    print(f"\n✅ Predictions saved to: {output_dir}")
    
    
    if len(disp_inputs) > 0:
        print("📊 Visualizing Best Detected Samples:")
        visualize_test_advanced(disp_inputs, disp_raw, disp_binary, disp_overlays)
    else:
        print("⚠️ Model predicted empty masks for the first few images.")


process_test_data(model, TEST_PATH, OUTPUT_DIR, max_display=5)


!zip -r inference_results.zip /kaggle/working/inference_results
print("\n🔗 inference_results.zip is ready in the Output sidebar!")

## 16. Quantitative Uncertainty Analysis: Model Confidence Distribution and Edge Ambiguity

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
import cv2
from tqdm import tqdm


TEST_PATH = '/kaggle/input/test-seg'
IMG_SIZE = (128, 128)

def analyze_test_uncertainty(model, source_dir, max_images=100):
    print(f"🚀 Analyzing Model Confidence on Test Data ({max_images} samples)...")
    
    image_files = [f for f in os.listdir(source_dir) if f.endswith(('.jpg', '.png'))][:max_images]
    
    all_confidences = [] 
    high_confidence_pixels = 0
    low_confidence_pixels = 0
    
   
    example_img = None
    example_uncertainty = None
    example_pred = None
    
    for img_name in tqdm(image_files):
       
        path = os.path.join(source_dir, img_name)
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        input_img = cv2.resize(img, IMG_SIZE).astype(np.float32) / 255.0
        
        
        
        pred_prob = model.predict(np.expand_dims(input_img, axis=0), verbose=0)
        pred_prob = np.squeeze(pred_prob) # (128, 128)
        
        
        flat_probs = pred_prob.flatten()
        sample_pixels = np.random.choice(flat_probs, size=1000) 
        all_confidences.extend(sample_pixels)
        
        
        if example_img is None and np.mean(pred_prob) > 0.05:
            example_img = input_img
            example_pred = (pred_prob > 0.5).astype(float)
            
            example_uncertainty = 1.0 - np.abs(2 * (pred_prob - 0.5))

    
    plt.figure(figsize=(18, 6))
    
    plt.subplot(1, 3, 1)
    sns.histplot(all_confidences, bins=50, kde=True, color="purple", stat="density")
    plt.title("Model Confidence Distribution (Test Set)", fontweight='bold')
    plt.xlabel("Pixel Probability (0=Background, 1=Object)")
    plt.ylabel("Density")
    plt.axvline(0.5, color='red', linestyle='--', label="Decision Boundary")
    plt.legend()
    

   
    if example_img is not None:
        plt.subplot(1, 3, 2)
        plt.imshow(example_pred, cmap='gray')
        plt.title("Predicted Binary Mask", fontweight='bold')
        plt.axis('off')
        
        plt.subplot(1, 3, 3)
        
        plt.imshow(example_uncertainty, cmap='hot', vmin=0, vmax=1)
        plt.title("Uncertainty Map (Heatmap)\nRed = High Uncertainty", fontweight='bold')
        plt.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print("\n✅ INTERPRETATION:")
    print("1. Histogram: You want peaks at 0.0 and 1.0. If many pixels are near 0.5, the model is 'confused'.")
    print("2. Uncertainty Map: Highlights the 'edges' of objects where the model struggles to decide.")


analyze_test_uncertainty(model, TEST_PATH, max_images=50)

## 17. Final Model Export: Archiving Performance Metrics and Paper-Ready Assets

In [ ]:
import os
import shutil
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.utils import plot_model
from sklearn.metrics import confusion_matrix
import cv2
from tqdm import tqdm


BASE_DIR = '/kaggle/working/paper_assets'
if os.path.exists(BASE_DIR):
    shutil.rmtree(BASE_DIR) 
os.makedirs(f"{BASE_DIR}/model_arch")
os.makedirs(f"{BASE_DIR}/training_metrics")
os.makedirs(f"{BASE_DIR}/validation_stats")
os.makedirs(f"{BASE_DIR}/test_visualizations")


sns.set_theme(style="whitegrid", context="paper", font_scale=1.5)
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'

print("📂 Created 'paper_assets' directory structure.")


print("🏗️ Generating Model Architecture Diagram...")
try:
    
    plot_model(
        model, 
        to_file=f"{BASE_DIR}/model_arch/model_architecture.png", 
        show_shapes=True, 
        show_layer_names=True,
        expand_nested=True,
        dpi=96
    )
    print("✅ Model architecture saved.")
except Exception as e:
    print(f"⚠️ Could not plot model (Graphviz missing?): {e}")


print("📈 Generating Training Graphs...")
if 'history' in globals():
    hist_df = pd.DataFrame(history.history)
    best_epoch = hist_df['val_loss'].idxmin()

    
    fig, axes = plt.subplots(1, 3, figsize=(24, 6))
    
    
    sns.lineplot(data=hist_df[['loss', 'val_loss']], ax=axes[0], linewidth=3)
    axes[0].set_title('Learning Curve (Loss)')
    axes[0].axvline(best_epoch, color='green', linestyle='--', alpha=0.5)
    
   
    sns.lineplot(data=hist_df[['accuracy', 'val_accuracy']], ax=axes[1], linewidth=3)
    axes[1].set_title('Pixel Accuracy')
    
    
    miou_keys = [k for k in hist_df.columns if 'mean_io_u' in k]
    if len(miou_keys) > 0:
        sns.lineplot(data=hist_df[miou_keys], ax=axes[2], linewidth=3)
        axes[2].set_title('mIoU Score')

    plt.tight_layout()
    plt.savefig(f"{BASE_DIR}/training_metrics/training_curves.png")
    plt.close()
    
    
    hist_df.to_csv(f"{BASE_DIR}/training_metrics/training_history.csv")
    print("✅ Training graphs and CSV saved.")


print("📊 Generating Confusion Matrix (Validation)...")

total_cm = np.zeros((2, 2))
steps = 10 
for i in range(steps):
    imgs, true_masks = val_generator[i]
    preds = model.predict(imgs, verbose=0)
    pred_bin = (preds > 0.5).astype(int).flatten()
    true_bin = (true_masks > 0.5).astype(int).flatten()
    total_cm += confusion_matrix(true_bin, pred_bin, labels=[0, 1])


plt.figure(figsize=(8, 6))
cm_norm = total_cm.astype('float') / total_cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', cbar=False,
            xticklabels=['Background', 'Object'], yticklabels=['Background', 'Object'])
plt.title('Pixel-wise Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.savefig(f"{BASE_DIR}/validation_stats/confusion_matrix.png")
plt.close()
print("✅ Confusion Matrix saved.")


print("🎨 Generating Test Set Visualizations...")
TEST_PATH = '/kaggle/input/test-seg'
test_files = os.listdir(TEST_PATH)[:50] 
count_saved = 0

for img_name in tqdm(test_files):
    if count_saved >= 5: break 
    
    path = os.path.join(TEST_PATH, img_name)
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    inp = cv2.resize(img, (128, 128))
    inp_tensor = np.expand_dims(inp.astype(np.float32) / 255.0, axis=0)
    
    pred = model.predict(inp_tensor, verbose=0)
    pred_mask = (pred > 0.5).astype(np.uint8).squeeze()
    
    
    if np.sum(pred_mask) > 100:
        count_saved += 1
        
        
        fig, ax = plt.subplots(1, 4, figsize=(20, 5))
        
        
        ax[0].imshow(inp)
        ax[0].set_title("Input")
        ax[0].axis('off')
        
       
        ax[1].imshow(pred_mask, cmap='gray')
        ax[1].set_title("Binary Mask")
        ax[1].axis('off')
        
        
        ax[2].imshow(pred.squeeze(), cmap='magma')
        ax[2].set_title("Confidence Heatmap")
        ax[2].axis('off')
        
       
        colored = np.zeros_like(inp)
        colored[:,:,1] = 255 # Green
        masked = cv2.bitwise_and(colored, colored, mask=pred_mask)
        overlay = cv2.addWeighted(inp, 0.7, masked, 0.3, 0)
        ax[3].imshow(overlay)
        ax[3].set_title("Overlay")
        ax[3].axis('off')
        
        plt.savefig(f"{BASE_DIR}/test_visualizations/prediction_{img_name}")
        plt.close()

print("✅ Test visualizations saved.")


print("📦 Zipping files for download...")
shutil.make_archive('/kaggle/working/paper_assets_download', 'zip', BASE_DIR)
print("\n=======================================================")
print("🎉 DONE! Download 'paper_assets_download.zip' from the Output sidebar.")
print("=======================================================")